In [ ]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import csv
import time

PARK_ID = 160
CSV_FILE = "efteling_parkinfo_all_years.csv"

# ------------------------------------
# Build day URL
# ------------------------------------
def build_url(date_obj):
    return f"https://queue-times.com/parks/{PARK_ID}/calendar/{date_obj.year}/{date_obj.month:02d}/{date_obj.day:02d}"

# ------------------------------------
# Find panel by fuzzy <h2> match
# ------------------------------------
def extract_panel_by_title(soup, *keywords):
    for h2 in soup.find_all("h2"):
        text = h2.get_text(strip=True).lower()
        if any(kw.lower() in text for kw in keywords):
            return h2.find_parent("div", class_="panel")
    return None

# ------------------------------------
# Extract text from panel rows
# ------------------------------------
def extract_value(panel, key_substring):
    key_substring = key_substring.lower()
    for block in panel.select("div.panel-block"):
        spans = block.find_all("span")
        if len(spans) < 2:
            continue
        key = spans[0].get_text(strip=True).lower()
        val = spans[1].get_text(strip=True)
        if key_substring in key:
            return val
    return None

# ------------------------------------
# Scrape one day
# ------------------------------------
def scrape_day(d):
    url = build_url(d)
    res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})

    if res.status_code != 200:
        print(f"⚠️ Failed HTTP fetch {d}: {res.status_code}")
        return None

    soup = BeautifulSoup(res.text, "html.parser")

    # ===========
    # CROWD INFO
    # ===========
    panel_info = extract_panel_by_title(soup, "information")

    crowd_percent = None
    crowd_label = None
    prediction_error = None

    if panel_info:
        rows = panel_info.select("div.panel-block span")
        pairs = list(zip(rows[0::2], rows[1::2]))

        for left, right in pairs:
            key = left.get_text(strip=True).lower()
            val = right.get_text(strip=True)
            if key == "crowd level" and "%" in val:
                crowd_percent = val
            elif key == "crowd level":
                crowd_label = val
            elif "prediction error" in key:
                prediction_error = val

    # ===========
    # TEMPERATURE
    # ===========
    panel_temp = extract_panel_by_title(soup, "temperature")

    temp_actual = None
    temp_forecast = None

    if panel_temp:
        temp_actual = extract_value(panel_temp, "actual")
        temp_forecast = extract_value(panel_temp, "forecast")

    # ===========
    # PRECIPITATION
    # ===========
    panel_prec = extract_panel_by_title(soup, "precipitation")

    prec_actual = None
    prec_forecast = None

    if panel_prec:
        prec_actual = extract_value(panel_prec, "actual")
        prec_forecast = extract_value(panel_prec, "forecast")

    # ===========
    # WIND
    # ===========
    panel_wind = extract_panel_by_title(soup, "wind speed")

    wind_actual = None
    wind_forecast = None

    if panel_wind:
        wind_actual = extract_value(panel_wind, "actual")
        wind_forecast = extract_value(panel_wind, "forecast")

    # ===========
    # EVENTS
    # ===========
    panel_events = extract_panel_by_title(soup, "event")

    events = None
    if panel_events:
        names = []
        for block in panel_events.select("div.panel-block"):
            span = block.find("span")
            if span:
                name = span.get_text(strip=True)
                if name:
                    names.append(name)
        events = "; ".join(names) if names else None

    return {
    "date": d.strftime("%Y-%m-%d"),
    "crowd_percent": crowd_percent,
    "crowd_label": crowd_label,
    "temperature_forecast": temp_forecast,
    "temperature_actual": temp_actual,
    "intensity_forecast": prec_forecast,
    "intensity_actual": prec_actual,
    "wind_forecast": wind_forecast,
    "wind_actual": wind_actual,
    "events": events
}


# ===========
# Write CSV header
# ===========
with open(CSV_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "date",
        "crowd_percent",
        "crowd_label",
        "temperature_forecast",
        "temperature_actual",
        "intensity_forecast",
        "intensity_actual",
        "wind_forecast",
        "wind_actual",
        "events"
    ])


# ===========
# SCRAPE DATE RANGE
# ===========
start_date = datetime(2023, 1, 1)
end_date = datetime.today()

current = start_date
print(f"Scraping from {start_date.date()} → {end_date.date()}")

while current <= end_date:

    print(f"Scraping {current}…")
    data = scrape_day(current)

    if data:
        with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                data["date"],
                data["crowd_percent"],
                data["crowd_label"],
                data["temperature_forecast"],
                data["temperature_actual"],
                data["intensity_forecast"],
                data["intensity_actual"],
                data["wind_forecast"],
                data["wind_actual"],
                data["events"]
            ])
        print(f"✔ Saved data for {current.date()}")
    else:
        print(f"⚠️ Skipped {current.date()} (no data)")

    current += timedelta(days=1)
    time.sleep(0.4)   # be polite


Scraping from 2023-01-01 → 2025-11-23
Scraping 2023-01-01 00:00:00…
✔ Saved data for 2023-01-01
Scraping 2023-01-02 00:00:00…
✔ Saved data for 2023-01-02
Scraping 2023-01-03 00:00:00…
✔ Saved data for 2023-01-03
Scraping 2023-01-04 00:00:00…
✔ Saved data for 2023-01-04
Scraping 2023-01-05 00:00:00…
✔ Saved data for 2023-01-05
Scraping 2023-01-06 00:00:00…
✔ Saved data for 2023-01-06
Scraping 2023-01-07 00:00:00…
✔ Saved data for 2023-01-07
Scraping 2023-01-08 00:00:00…
✔ Saved data for 2023-01-08
Scraping 2023-01-09 00:00:00…
✔ Saved data for 2023-01-09
Scraping 2023-01-10 00:00:00…
✔ Saved data for 2023-01-10
Scraping 2023-01-11 00:00:00…
✔ Saved data for 2023-01-11
Scraping 2023-01-12 00:00:00…
✔ Saved data for 2023-01-12
Scraping 2023-01-13 00:00:00…
✔ Saved data for 2023-01-13
Scraping 2023-01-14 00:00:00…
✔ Saved data for 2023-01-14
Scraping 2023-01-15 00:00:00…
✔ Saved data for 2023-01-15
Scraping 2023-01-16 00:00:00…
✔ Saved data for 2023-01-16
Scraping 2023-01-17 00:00:00…
✔ Sa